# Ark+ 3D-Only Cyclic Pretraining
**GPU:** RTX 4060 &nbsp;|&nbsp; **Env:** ark3d &nbsp;|&nbsp; **Repo:** [ArnabmoyPaul/ark_3d_only](https://github.com/ArnabmoyPaul/ark_3d_only)

### What this notebook does
| Cell | Action |
|------|--------|
| 1 | Verify GPU + CUDA |
| 2 | Clone / pull repo from GitHub |
| 3 | Install dependencies |
| 4 | Verify all 6 datasets + task types |
| 5 | **Run 200-epoch 3D training** |
| 6 | Resume after interruption |
| 7 | Generate plots (PNG) |
| 8 | Print final results table |

---
### SOTA targets (goal: within 3%)
| Dataset | Task | SOTA |
|---------|------|------|
| OrganMNIST3D | 11-class ACC | 0.997 |
| NoduleMNIST3D | binary AUC | 0.863 |
| AdrenalMNIST3D | binary AUC | 0.874 |
| FractureMNIST3D | 3-class ACC | 0.714 |
| VesselMNIST3D | binary AUC | 0.914 |
| SynapseMNIST3D | binary AUC | 0.843 |

---
## Cell 1 — Verify GPU

In [ ]:
import torch

print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()}')

if torch.cuda.is_available():
    print(f'GPU      : {torch.cuda.get_device_name(0)}')
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM     : {vram:.1f} GB')
    if vram < 6:
        print('⚠️  Less than 6 GB VRAM — reduce --batch_size to 32')
    else:
        print('✓ VRAM OK for batch_size=64')
else:
    print('⚠️  No GPU found — training will use CPU (very slow)')
    print('   Change --device cuda → --device cpu in Cell 5')

---
## Cell 2 — Clone / Pull Repo from GitHub

In [ ]:
import os, sys, subprocess

# ── EDIT THIS: where to put the project on your machine ──────────────────────
WORK_DIR = r'C:\Users\Arnabmoy\Desktop\ark_3d_only'
# ─────────────────────────────────────────────────────────────────────────────

REPO_URL = 'https://github.com/ArnabmoyPaul/ark_3d_only.git'
REPO_DIR = WORK_DIR

os.makedirs(os.path.dirname(WORK_DIR), exist_ok=True)

if not os.path.exists(os.path.join(REPO_DIR, '.git')):
    print('Cloning repo...')
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
    print('✓ Cloned successfully')
else:
    print('Repo exists — pulling latest changes...')
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
    print('✓ Up to date')

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print(f'\nWorking dir : {os.getcwd()}')
py_files = sorted([f for f in os.listdir('.') if f.endswith('.py')])
print(f'Python files: {py_files}')
other_files = [f for f in os.listdir('.') if not f.endswith('.py') and not f.startswith('.')]
print(f'Other files : {other_files}')

---
## Cell 3 — Install / Verify Dependencies

In [ ]:
import subprocess, sys

print('Checking / installing dependencies...')
print('=' * 50)

# Install from requirements.txt
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt', '-q'],
    capture_output=True, text=True
)
if result.returncode != 0:
    print('⚠️  Some packages may have failed:')
    print(result.stderr[-500:])
else:
    print('✓ requirements.txt installed')

print()

# Verify each import
checks = [
    ('torch',     lambda: __import__('torch').__version__),
    ('timm',      lambda: __import__('timm').__version__),
    ('medmnist',  lambda: __import__('medmnist').__version__),
    ('einops',    lambda: __import__('einops').__version__),
    ('sklearn',   lambda: __import__('sklearn').__version__),
    ('matplotlib',lambda: __import__('matplotlib').__version__),
    ('yaml',      lambda: __import__('yaml').__version__),
    ('scipy',     lambda: __import__('scipy').__version__),
]

all_ok = True
for name, ver_fn in checks:
    try:
        v = ver_fn()
        print(f'  ✓ {name:<15s} {v}')
    except Exception as e:
        print(f'  ✗ {name:<15s} MISSING — {e}')
        all_ok = False

print()
if all_ok:
    print('✓ All dependencies OK — ready to train!')
else:
    print('✗ Some dependencies missing. Re-run this cell or run setup_env.bat')

---
## Cell 4 — Verify Datasets & Task Types
This confirms all 6 datasets resolve to the correct loss function before training starts.

In [ ]:
from utils import get_config
from engine_3d import _is_multiclass, _is_binary, _criterion
import torch

cfg = get_config('datasets_config_3d.yaml')

SOTA = {
    'OrganMNIST3D':   0.997,
    'NoduleMNIST3D':  0.863,
    'AdrenalMNIST3D': 0.874,
    'FractureMNIST3D':0.714,
    'VesselMNIST3D':  0.914,
    'SynapseMNIST3D': 0.843,
}

print(f'{"Dataset":<22s} {"n_cls":>5s}  {"Loss":<25s} {"SOTA":>6s}')
print('-' * 65)

all_ok = True
for name, info in cfg.items():
    tt   = info['task_type']
    n    = len(info['diseases'])
    loss = _criterion(tt).__class__.__name__
    mc   = _is_multiclass(tt)
    bi   = _is_binary(tt)
    sota = SOTA.get(name, 0.0)

    # Sanity checks
    ok = True
    if mc and n == 1:
        ok = False; print(f'  ✗ {name}: multi-class but n_cls=1!')
    if bi and n != 1:
        ok = False; print(f'  ✗ {name}: binary but n_cls={n}!')

    status = '✓' if ok else '✗'
    print(f'{status} {name:<20s} {n:>5d}  {loss:<25s} {sota:>6.3f}')
    if not ok:
        all_ok = False

print()
if all_ok:
    print('✓ All 6 datasets configured correctly — safe to run training')
else:
    print('✗ Config issues found — check datasets_config_3d.yaml')

---
## Cell 5 — Run 200-Epoch Training

Output streams live to this cell AND to `Logs/run04_3donly_*.txt`.

**Expected time:** ~3–4 hours on RTX 4060 (200 epochs × 6 datasets × ~1000 samples).

**Test evaluation** printed every 5 epochs showing mAUC per dataset vs SOTA.

> 💡 Don't close VS Code — let it run. If interrupted, use Cell 6 to resume.

In [ ]:
import subprocess, sys, os
from datetime import datetime

# ── Training configuration ────────────────────────────────────────────────────
EXP_NAME   = 'run04_3donly'
EPOCHS     = 200
BATCH_SIZE = 64       # reduce to 32 if CUDA OOM
LR         = '1e-3'
MOMENTUM   = '0.9'   # original Ark+ value — correct for small 3D datasets
WARMUP     = 10
TEST_EVERY = 5        # evaluate on test set every N epochs
MODEL      = 'swin_tiny'
DEVICE     = 'cuda'
WORKERS    = 4
# ─────────────────────────────────────────────────────────────────────────────

os.makedirs('Logs', exist_ok=True)
ts      = datetime.now().strftime('%Y%m%d_%H%M%S')
logfile = f'Logs/{EXP_NAME}_{ts}.txt'

cmd = [
    sys.executable, 'main_3d.py',
    '--model',            MODEL,
    '--pretrain_epochs',  str(EPOCHS),
    '--batch_size',       str(BATCH_SIZE),
    '--lr',               LR,
    '--momentum_teacher', MOMENTUM,
    '--warmup_epochs',    str(WARMUP),
    '--test_epoch',       str(TEST_EVERY),
    '--exp_name',         EXP_NAME,
    '--device',           DEVICE,
    '--workers',          str(WORKERS),
]

print('Command:', ' '.join(cmd[1:]))
print(f'Log file: {logfile}')
print('=' * 70)

log_lines = []
process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True, bufsize=1,
    encoding='utf-8', errors='replace',
)

with open(logfile, 'w', encoding='utf-8') as log_f:
    for line in process.stdout:
        print(line, end='', flush=True)
        log_f.write(line)
        log_f.flush()

process.wait()
print('=' * 70)
if process.returncode == 0:
    print(f'✓ Training complete!  Log saved: {logfile}')
else:
    print(f'✗ Training exited with code {process.returncode}')
    print(f'  Check log: {logfile}')

---
## Cell 6 — Resume (if interrupted)
Continues from the last saved checkpoint. `--exp_name` must match Cell 5.

In [ ]:
import subprocess, sys, os
from datetime import datetime

# ── Must match Cell 5 exactly ────────────────────────────────────────────────
EXP_NAME   = 'run04_3donly'
EPOCHS     = 200
BATCH_SIZE = 64
LR         = '1e-3'
MOMENTUM   = '0.9'
WARMUP     = 10
TEST_EVERY = 5
MODEL      = 'swin_tiny'
DEVICE     = 'cuda'
WORKERS    = 4
# ─────────────────────────────────────────────────────────────────────────────

os.makedirs('Logs', exist_ok=True)
ts      = datetime.now().strftime('%Y%m%d_%H%M%S')
logfile = f'Logs/{EXP_NAME}_resume_{ts}.txt'

cmd = [
    sys.executable, 'main_3d.py',
    '--model',            MODEL,
    '--pretrain_epochs',  str(EPOCHS),
    '--batch_size',       str(BATCH_SIZE),
    '--lr',               LR,
    '--momentum_teacher', MOMENTUM,
    '--warmup_epochs',    str(WARMUP),
    '--test_epoch',       str(TEST_EVERY),
    '--exp_name',         EXP_NAME,
    '--device',           DEVICE,
    '--workers',          str(WORKERS),
    '--resume',
]

print('Resuming:', ' '.join(cmd[1:]))
print(f'Log file : {logfile}')
print('=' * 70)

process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True, bufsize=1,
    encoding='utf-8', errors='replace',
)

with open(logfile, 'w', encoding='utf-8') as log_f:
    for line in process.stdout:
        print(line, end='', flush=True)
        log_f.write(line)
        log_f.flush()

process.wait()
print('=' * 70)
print(f'Exited with code: {process.returncode}')

---
## Cell 7 — Generate Plots
Saves 5 PNG files to `Plots/`. Click any file in the VS Code Explorer panel to preview.

In [ ]:
import subprocess, sys

EXP_NAME = 'run04_3donly'   # must match Cell 5

result = subprocess.run(
    [sys.executable, 'plot_results.py',
     '--exp_name', EXP_NAME,
     '--out_dir',  'Plots'],
    capture_output=False, text=True
)

print('\nPlots saved to Plots/ — open in VS Code Explorer to preview.')

---
## Cell 8 — Print Final Results Table
Parses the results file and shows the final student mAUC vs SOTA for all 6 datasets.

In [ ]:
import os, re, glob

EXP_NAME   = 'run04_3donly'
MODEL_NAME = 'swin_tiny'

SOTA = {
    'OrganMNIST3D':   0.997,
    'NoduleMNIST3D':  0.863,
    'AdrenalMNIST3D': 0.874,
    'FractureMNIST3D':0.714,
    'VesselMNIST3D':  0.914,
    'SynapseMNIST3D': 0.843,
}
ALL_DS = list(SOTA.keys())

# Find results file
patterns = [
    f'Outputs/{MODEL_NAME}_{EXP_NAME}/*results.txt',
    'Outputs/**/*results*.txt',
]
results_file = None
for pat in patterns:
    m = glob.glob(pat, recursive=True)
    if m:
        results_file = sorted(m)[-1]
        break

if not results_file:
    print('No results file found yet — run at least 5 epochs (first test_epoch).')
else:
    print(f'Reading: {results_file}\n')

    # Parse all eval blocks
    blocks = []
    cur = None
    with open(results_file, errors='ignore') as f:
        for line in f:
            m_ep = re.search(r'Epoch\s+(\d+)', line)
            if m_ep and ('─' in line or ':' in line):
                if cur and cur['ds']:
                    blocks.append(cur)
                cur = {'epoch': int(m_ep.group(1)), 'ds': {}}
                continue
            if cur is None:
                continue
            for ds in ALL_DS:
                if ds in line:
                    s = re.search(r'S:.*?mAUC=([\d.]+)', line) or re.search(r'S:.*?ACC=([\d.]+)', line)
                    t = re.search(r'T:.*?mAUC=([\d.]+)', line) or re.search(r'T:.*?ACC=([\d.]+)', line)
                    if s:
                        cur['ds'][ds] = {'s': float(s.group(1)),
                                         't': float(t.group(1)) if t else float(s.group(1))}
    if cur and cur['ds']:
        blocks.append(cur)

    if not blocks:
        print('No eval blocks found in results file yet.')
    else:
        final = blocks[-1]
        print(f'Results at Epoch {final["epoch"]}:')
        print()
        print(f'{"Dataset":<22s} {"Student":>8s} {"Teacher":>8s} {"SOTA":>6s} {"Gap":>8s} Status')
        print('-' * 70)

        total_s, total_sota, n = 0, 0, 0
        for ds in ALL_DS:
            sota_v = SOTA[ds]
            if ds in final['ds']:
                sv   = final['ds'][ds]['s']
                tv   = final['ds'][ds]['t']
                gap  = sv - sota_v
                if   gap >= -0.01: status = '✓ ≤1%'
                elif gap >= -0.03: status = '◑ ≤3%'
                elif gap >= -0.10: status = '● ≤10%'
                else:              status = '✗ >10%'
                print(f'{ds:<22s} {sv:>8.4f} {tv:>8.4f} {sota_v:>6.3f} {gap:>+8.4f} {status}')
                total_s += sv; total_sota += sota_v; n += 1
            else:
                print(f'{ds:<22s} {"pending":>8s}')

        if n:
            print('-' * 70)
            print(f'{"Mean":<22s} {total_s/n:>8.4f} {"":>8s} {total_sota/n:>6.3f} {(total_s-total_sota)/n:>+8.4f}')
        print()

        # Eval history
        print('mAUC history across eval epochs:')
        ep_list = [b['epoch'] for b in blocks]
        print(f'{"Dataset":<22s}', end='')
        for ep in ep_list[-10:]:   # last 10 evals
            print(f'{ep:>7d}', end='')
        print()
        for ds in ALL_DS:
            print(f'{ds:<22s}', end='')
            for b in blocks[-10:]:
                val = b['ds'].get(ds, {}).get('s')
                print(f'{val:>7.4f}' if val else f'{"?":>7s}', end='')
            print()

---
## Cell 9 — Quick Sanity Test (CPU, 3 epochs)
Run this **before** Cell 5 to verify the full pipeline works without waiting for GPU training.

In [ ]:
import subprocess, sys

print('Quick sanity test: 2 datasets × 3 epochs × CPU')
print('Expected: ~2 minutes')
print('=' * 50)

cmd = [
    sys.executable, 'main_3d.py',
    '--datasets',         'NoduleMNIST3D', 'SynapseMNIST3D',
    '--model',            'swin_tiny',
    '--pretrain_epochs',  '3',
    '--batch_size',       '16',
    '--lr',               '1e-3',
    '--momentum_teacher', '0.9',
    '--warmup_epochs',    '0',
    '--test_epoch',       '1',
    '--exp_name',         'sanity_test',
    '--device',           'cpu',
    '--workers',          '0',
]

process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True, bufsize=1,
    encoding='utf-8', errors='replace',
)
for line in process.stdout:
    print(line, end='', flush=True)
process.wait()

print('=' * 50)
if process.returncode == 0:
    print('✓ Sanity test passed — pipeline is working correctly!')
    print('  Now run Cell 5 for full training.')
else:
    print(f'✗ Sanity test failed (code {process.returncode})')
    print('  Check error messages above before running Cell 5.')